In [1]:
import numpy as np
import pandas as pd
import re
from tqdm import tqdm, trange
from scipy.special import comb, perm
from itertools import combinations

from scipy.stats import pearsonr, spearmanr

import matplotlib.pyplot as plt
import sklearn
from sklearn import preprocessing
from scipy.stats import pearsonr, spearmanr 

plt.style.use('default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

maxdist阈值的确定（4-way正样本）：
1. 计算四个点两两之间的空间距离的`平均值`作为四点间的平均距离
2. 计算不同的maxdist三点在`所有细胞中共定位的比例`与`MCI freq`的`相关系数`：100/150/200/250/300/350/400/450/500nm
3. 取`相关系数最大`时对应的maxdist作为最终阈值

In [2]:
def calc_colocalization_ratio(bin_1, bin_2, bin_3, bin_4, coor_dict, max_dist_threshold=250):

    haploids = list(coor_dict[bin_1])
    dist_avg_list = []

    for haploid in haploids:
        coor_bin_1 = coor_dict[bin_1][haploid]
        coor_bin_2 = coor_dict[bin_2][haploid]
        coor_bin_3 = coor_dict[bin_3][haploid]
        coor_bin_4 = coor_dict[bin_4][haploid]

        dist_12 = np.linalg.norm(coor_bin_1 - coor_bin_2)
        dist_13 = np.linalg.norm(coor_bin_1 - coor_bin_3)
        dist_14 = np.linalg.norm(coor_bin_1 - coor_bin_4)
        dist_23 = np.linalg.norm(coor_bin_2 - coor_bin_3)
        dist_24 = np.linalg.norm(coor_bin_2 - coor_bin_4)
        dist_34 = np.linalg.norm(coor_bin_3 - coor_bin_4)

        dist_avg = np.mean([dist_12, dist_13, dist_14, dist_23, dist_24, dist_34])

        dist_avg_list.append(dist_avg)

    dist_avg_list = np.array(dist_avg_list)

    colocalization_ratio = dist_avg_list[dist_avg_list < max_dist_threshold].shape[0] / len(haploids)

    return colocalization_ratio

In [20]:
chroms = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10',
            'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19']

df_coloc_ratio_vs_freq_corr = pd.DataFrame(columns=['chrom', 'max_dist_coloc_ratio', 'pearson_R', 'pearson_P', 'spearman_R', 'spearman_P'])

for chrom in chroms:
    print(f'{chrom} ...')
    try:
        df_chrom_freq = pd.read_csv(f'./MCI_Freq_vs_Colocalization_ratio/mESC_25Kb_FISH_MCI_Freq_{chrom}.csv', sep='\t')
        df_chrom_coloc_ratio = pd.read_csv(f'./MCI_Freq_vs_Colocalization_ratio/mESC_25Kb_MCI_Freq_vs_Colocalization_ratio_{chrom}.csv', sep='\t')
        df_chrom_coloc_ratio['freq'] = df_chrom_freq['freq']
        # df_chrom_coloc_ratio = df_chrom_coloc_ratio.loc[df_chrom_coloc_ratio.freq > 0]
        # df_chrom_coloc_ratio = df_chrom_coloc_ratio.loc[df_chrom_coloc_ratio.maxdist_300nm_colocalization_ratio > 0]

        max_dist_ratio = [
                    'maxdist_50nm_colocalization_ratio',
                    'maxdist_100nm_colocalization_ratio',
                    'maxdist_150nm_colocalization_ratio',
                    'maxdist_200nm_colocalization_ratio',
                    'maxdist_250nm_colocalization_ratio',
                    'maxdist_300nm_colocalization_ratio',
                    'maxdist_350nm_colocalization_ratio',
                    'maxdist_400nm_colocalization_ratio',
                    'maxdist_450nm_colocalization_ratio',
                    'maxdist_500nm_colocalization_ratio']
        max_dist_list = []
        p_r_list, p_p_list, s_r_list, s_p_list = [], [], [], []
        for max_dist in max_dist_ratio:
            df_ = df_chrom_coloc_ratio.loc[(df_chrom_coloc_ratio['freq'] > 0) & (df_chrom_coloc_ratio[max_dist] > 0)]

            p_r, p_p = pearsonr(df_['freq'], df_[max_dist])
            s_r, s_p = spearmanr(df_['freq'], df_[max_dist])
            # p_r, p_p = pearsonr(df_chrom_coloc_ratio['freq'], df_chrom_coloc_ratio[max_dist])
            # s_r, s_p = spearmanr(df_chrom_coloc_ratio['freq'], df_chrom_coloc_ratio[max_dist])
            p_r_list.append(p_r)
            p_p_list.append(p_p)
            s_r_list.append(s_r)
            s_p_list.append(s_p)
            max_dist_list.append(max_dist)


        df_tmp = pd.DataFrame({
            'chrom': [chrom] * 10, 
            'max_dist_coloc_ratio': max_dist_list,
            'pearson_R': p_r_list,
            'pearson_P': p_p_list,
            'spearman_R': s_r_list,
            'spearman_P': s_p_list
        })

        df_coloc_ratio_vs_freq_corr = pd.concat([df_coloc_ratio_vs_freq_corr, df_tmp])
    except:
        continue
df_coloc_ratio_vs_freq_corr.to_csv('Correlation_between_Coloc_ratio_and_Freq.txt', header=True, index=False, sep='\t')

chr1 ...


/data/xujs/miniconda3/lib/python3.7/site-packages/scipy/stats/stats.py:4023: PearsonRConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(PearsonRConstantInputWarning())
/data/xujs/miniconda3/lib/python3.7/site-packages/scipy/stats/stats.py:4484: SpearmanRConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(SpearmanRConstantInputWarning())


chr2 ...
chr3 ...
chr4 ...
chr5 ...
chr6 ...
chr7 ...
chr8 ...
chr9 ...
chr10 ...
chr11 ...
chr12 ...
chr13 ...
chr14 ...
chr15 ...
chr16 ...
chr17 ...
chr18 ...
chr19 ...


In [14]:
chrom = 'chr1'

df_chrom_freq = pd.read_csv(f'./MCI_Freq_vs_Colocalization_ratio/mESC_25Kb_FISH_MCI_Freq_{chrom}.csv', sep='\t')
df_chrom_coloc_ratio = pd.read_csv(f'./MCI_Freq_vs_Colocalization_ratio/mESC_25Kb_MCI_Freq_vs_Colocalization_ratio_{chrom}.csv', sep='\t')
df_chrom_coloc_ratio['freq'] = df_chrom_freq['freq']
# df_chrom_coloc_ratio = df_chrom_coloc_ratio.loc[df_chrom_coloc_ratio.freq > 0]
# df_chrom_coloc_ratio = df_chrom_coloc_ratio.loc[df_chrom_coloc_ratio.maxdist_300nm_colocalization_ratio > 0]

max_dist_ratio = [
        'maxdist_50nm_colocalization_ratio',
        'maxdist_100nm_colocalization_ratio',
        'maxdist_150nm_colocalization_ratio',
        'maxdist_200nm_colocalization_ratio',
        'maxdist_250nm_colocalization_ratio',
        'maxdist_300nm_colocalization_ratio',
        'maxdist_350nm_colocalization_ratio',
        'maxdist_400nm_colocalization_ratio',
        'maxdist_450nm_colocalization_ratio',
        'maxdist_500nm_colocalization_ratio']
max_dist_list = []
p_r_list, p_p_list, s_r_list, s_p_list = [], [], [], []

In [15]:
df_chrom_coloc_ratio

,bin1,bin2,bin3,bin4,freq,maxdist_50nm_colocalization_ratio,maxdist_100nm_colocalization_ratio,maxdist_150nm_colocalization_ratio,maxdist_200nm_colocalization_ratio,maxdist_250nm_colocalization_ratio,maxdist_300nm_colocalization_ratio,maxdist_350nm_colocalization_ratio,maxdist_400nm_colocalization_ratio,maxdist_450nm_colocalization_ratio,maxdist_500nm_colocalization_ratio
0,chr1:135600000,chr1:135625000,chr1:135650000,chr1:135675000,0,0.000000,0.009217,0.042627,0.096774,0.137097,0.169355,0.195853,0.206221,0.215438,0.224654
1,chr1:135600000,chr1:135625000,chr1:135650000,chr1:135700000,0,0.000000,0.002304,0.017281,0.059908,0.107143,0.158986,0.179724,0.200461,0.210829,0.218894
2,chr1:135600000,chr1:135625000,chr1:135650000,chr1:135725000,0,0.000000,0.000000,0.019585,0.057604,0.107143,0.139401,0.170507,0.191244,0.203917,0.213134
3,chr1:135600000,chr1:135625000,chr1:135650000,chr1:135750000,0,0.000000,0.000000,0.012673,0.047235,0.092166,0.125576,0.152074,0.168203,0.184332,0.193548
4,chr1:135600000,chr1:135625000,chr1:135650000,chr1:135775000,0,0.000000,0.000000,0.012673,0.047235,0.093318,0.135945,0.168203,0.191244,0.205069,0.210829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487630,chr1:136975000,chr1:137000000,chr1:137025000,chr1:137050000,0,0.000000,0.005760,0.054147,0.152074,0.233871,0.289171,0.328341,0.353687,0.365207,0.376728
487631,chr1:136975000,chr1:137000000,chr1:137025000,chr1:137075000,0,0.000000,0.003456,0.059908,0.150922,0.254608,0.306452,0.354839,0.379032,0.395161,0.402074
487632,chr1:136975000,chr1:137000000,chr1:137050000,chr1:137075000,0,0.000000,0.002304,0.041475,0.127880,0.215438,0.275346,0.317972,0.334101,0.353687,0.364055
487633,chr1:136975000,chr1:137025000,chr1:137050000,chr1:137075000,0,0.001152,0.003456,0.041475,0.133641,0.243088,0.289171,0.322581,0.345622,0.361751,0.365207


In [16]:
df_ = df_chrom_coloc_ratio.loc[(df_chrom_coloc_ratio['freq'] > 0) & (df_chrom_coloc_ratio[max_dist] > 0)]
p_r, p_p = pearsonr(df_['freq'], df_[max_dist])
s_r, s_p = spearmanr(df_['freq'], df_[max_dist])

In [18]:
df_chrom_coloc_ratio.loc[(df_chrom_coloc_ratio['freq'] > 0) & (df_chrom_coloc_ratio[max_dist] > 0)]

,bin1,bin2,bin3,bin4,freq,maxdist_50nm_colocalization_ratio,maxdist_100nm_colocalization_ratio,maxdist_150nm_colocalization_ratio,maxdist_200nm_colocalization_ratio,maxdist_250nm_colocalization_ratio,maxdist_300nm_colocalization_ratio,maxdist_350nm_colocalization_ratio,maxdist_400nm_colocalization_ratio,maxdist_450nm_colocalization_ratio,maxdist_500nm_colocalization_ratio
1711,chr1:135600000,chr1:135650000,chr1:135700000,chr1:135775000,1,0.0,0.000000,0.004608,0.031106,0.092166,0.147465,0.184332,0.198157,0.213134,0.217742
1714,chr1:135600000,chr1:135650000,chr1:135700000,chr1:135850000,1,0.0,0.000000,0.003456,0.031106,0.077189,0.123272,0.175115,0.200461,0.217742,0.235023
1715,chr1:135600000,chr1:135650000,chr1:135700000,chr1:135875000,1,0.0,0.000000,0.004608,0.029954,0.074885,0.130184,0.167051,0.201613,0.221198,0.237327
1722,chr1:135600000,chr1:135650000,chr1:135700000,chr1:136050000,1,0.0,0.000000,0.000000,0.006912,0.048387,0.096774,0.148618,0.194700,0.207373,0.224654
1765,chr1:135600000,chr1:135650000,chr1:135725000,chr1:135775000,1,0.0,0.000000,0.005760,0.027650,0.088710,0.145161,0.173963,0.197005,0.210829,0.216590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487540,chr1:136875000,chr1:136925000,chr1:137000000,chr1:137050000,1,0.0,0.000000,0.012673,0.047235,0.091014,0.148618,0.192396,0.211982,0.228111,0.231567
487550,chr1:136875000,chr1:136950000,chr1:137000000,chr1:137050000,1,0.0,0.001152,0.013825,0.061060,0.133641,0.202765,0.243088,0.277650,0.292627,0.306452
487585,chr1:136900000,chr1:136950000,chr1:137000000,chr1:137050000,1,0.0,0.000000,0.014977,0.080645,0.153226,0.225806,0.268433,0.298387,0.320276,0.330645
487586,chr1:136900000,chr1:136950000,chr1:137000000,chr1:137075000,1,0.0,0.000000,0.012673,0.074885,0.156682,0.241935,0.283410,0.307604,0.337558,0.346774
